# PyTorch Basics: Autograd - Automatic Differentiation

Welcome to the second PyTorch tutorial! This notebook covers:
- What is automatic differentiation?
- Computational graphs
- Gradient computation with `.backward()`
- Gradient accumulation and zeroing
- Disabling gradient tracking

## What is Autograd?

**Autograd** is PyTorch's automatic differentiation engine. It powers neural network training by:
- Automatically computing gradients
- Building computational graphs dynamically
- Enabling backpropagation

```
Forward Pass          Backward Pass
────────────────────────────────────
x → f(x) → y        y → ∂y/∂x → x.grad
```

The key idea: PyTorch tracks all operations on tensors that have `requires_grad=True` and can automatically compute gradients.


In [ ]:
import torch
import numpy as np

print(f"PyTorch version: {torch.__version__}")


## 1. Basic Gradient Computation

Let's start with a simple example: computing the gradient of y = x²


In [ ]:
# Create a tensor with gradient tracking enabled
x = torch.tensor(3.0, requires_grad=True)
print(f"x = {x}")
print(f"x.requires_grad = {x.requires_grad}")
print()

# Forward pass: compute y = x²
y = x ** 2
print(f"y = x² = {y}")
print(f"y.requires_grad = {y.requires_grad}")
print()

# Backward pass: compute gradient dy/dx
y.backward()

# The gradient dy/dx = 2x is stored in x.grad
print(f"dy/dx = 2x = 2*3 = {x.grad}")
print(f"Computed gradient: {x.grad}")


## 2. Computational Graphs

PyTorch builds a dynamic computational graph during the forward pass.

```
Example: z = (x + y) * (y + 2)

    x          y
     \        / \
      \      /   \
       \    /     \
        add      add
         \       /
          \     /
           \   /
            mul
             |
             z

Each operation creates a node in the graph.
Gradients flow backwards through this graph.
```


In [ ]:
# More complex example
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

# Forward pass
z = x**2 + y**3
print(f"x = {x}, y = {y}")
print(f"z = x² + y³ = {z}")
print()

# Backward pass
z.backward()

# Gradients
print(f"∂z/∂x = 2x = 2*2 = {x.grad}")
print(f"∂z/∂y = 3y² = 3*9 = {y.grad}")


## 3. Vector and Matrix Gradients


In [ ]:
# Vector example
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2
z = y.sum()  # Must reduce to scalar for backward()

print(f"x = {x}")
print(f"y = x² = {y}")
print(f"z = sum(y) = {z}")
print()

z.backward()
print(f"∂z/∂x = 2x = {x.grad}")
print()

# Matrix example
X = torch.randn(2, 3, requires_grad=True)
Y = X * 2
Z = Y.mean()

print(f"X shape: {X.shape}")
print(f"Z = {Z}")

Z.backward()
print(f"X.grad shape: {X.grad.shape}")
print(f"X.grad:\n{X.grad}")


## 4. Gradient Accumulation

⚠️ **Important**: Gradients accumulate by default!


In [ ]:
x = torch.tensor(3.0, requires_grad=True)

# First computation
y = x ** 2
y.backward()
print(f"After first backward: x.grad = {x.grad}")

# Second computation (gradient accumulates!)
y = x ** 3
y.backward()
print(f"After second backward: x.grad = {x.grad}")
print(f"Expected 3x² = {3 * x.item()**2}, but got accumulated value")
print()

# Solution: Zero gradients before each backward pass
x.grad.zero_()
y = x ** 3
y.backward()
print(f"After zeroing and backward: x.grad = {x.grad}")


## 5. Disabling Gradient Tracking

Sometimes you don't need gradients (e.g., during inference).


In [ ]:
x = torch.tensor(3.0, requires_grad=True)

# Method 1: torch.no_grad() context manager
with torch.no_grad():
    y = x ** 2
    print(f"Inside no_grad: y.requires_grad = {y.requires_grad}")

# Method 2: .detach()
y = (x ** 2).detach()
print(f"Using detach: y.requires_grad = {y.requires_grad}")

# Method 3: .requires_grad_(False)
x.requires_grad_(False)
y = x ** 2
print(f"After requires_grad_(False): y.requires_grad = {y.requires_grad}")

# Re-enable
x.requires_grad_(True)
y = x ** 2
print(f"After requires_grad_(True): y.requires_grad = {y.requires_grad}")


## 6. Gradient of Non-Scalar Outputs

For non-scalar outputs, you need to provide a gradient argument.


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2

print(f"y = {y}")
print(f"y is not a scalar, shape: {y.shape}")
print()

# Provide gradient for each output
gradient = torch.tensor([1.0, 1.0, 1.0])
y.backward(gradient=gradient)
print(f"x.grad with gradient=[1,1,1]: {x.grad}")
print()

# Reset
x.grad.zero_()
gradient = torch.tensor([0.1, 1.0, 100.0])
y.backward(gradient=gradient)
print(f"x.grad with gradient=[0.1,1,100]: {x.grad}")


## 7. Common Activation Functions and Their Gradients


In [ ]:
import torch.nn.functional as F

x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0], requires_grad=True)

# ReLU
y = F.relu(x)
print(f"ReLU({x.data}) = {y}")
y.sum().backward()
print(f"ReLU gradient: {x.grad}")
print()

# Sigmoid
x.grad.zero_()
y = torch.sigmoid(x)
print(f"Sigmoid({x.data}) = {y}")
y.sum().backward()
print(f"Sigmoid gradient: {x.grad}")
print()

# Tanh
x.grad.zero_()
y = torch.tanh(x)
print(f"Tanh({x.data}) = {y}")
y.sum().backward()
print(f"Tanh gradient: {x.grad}")


## 8. Practical Example: Linear Regression

Let's implement gradient descent manually using autograd.


In [ ]:
# Generate synthetic data: y = 3x + 2 + noise
torch.manual_seed(42)
X_train = torch.randn(100, 1) * 10
y_train = 3 * X_train + 2 + torch.randn(100, 1) * 2

# Initialize parameters
w = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

learning_rate = 0.01
epochs = 100

# Training loop
for epoch in range(epochs):
    # Forward pass
    y_pred = X_train * w + b
    
    # Compute loss (MSE)
    loss = ((y_pred - y_train) ** 2).mean()
    
    # Backward pass
    loss.backward()
    
    # Update parameters (no gradient tracking)
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
    
    # Zero gradients
    w.grad.zero_()
    b.grad.zero_()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}, w = {w.item():.4f}, b = {b.item():.4f}")

print(f"\nFinal parameters: w = {w.item():.4f}, b = {b.item():.4f}")
print(f"True parameters: w = 3.0, b = 2.0")


## 9. Higher-Order Gradients

You can compute gradients of gradients (second derivatives).


In [ ]:
x = torch.tensor(2.0, requires_grad=True)

# First derivative
y = x ** 3
y.backward(create_graph=True)  # Keep graph for second derivative
print(f"y = x³")
print(f"dy/dx = 3x² = {x.grad}")

# Second derivative
x.grad.backward()
print(f"d²y/dx² = 6x = {x.grad}")  # This will show accumulated gradient

# Clean example
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3
grad1 = torch.autograd.grad(y, x, create_graph=True)[0]
grad2 = torch.autograd.grad(grad1, x)[0]
print(f"\nUsing torch.autograd.grad:")
print(f"First derivative: {grad1}")
print(f"Second derivative: {grad2}")


## 📝 Summary

You've learned:

✅ What autograd is and how it works  
✅ Creating computational graphs  
✅ Computing gradients with `.backward()`  
✅ Gradient accumulation and zeroing  
✅ Disabling gradient tracking  
✅ Gradients for vectors and matrices  
✅ Implementing gradient descent manually  

### Key Points to Remember

1. **Always zero gradients** before backward pass: `x.grad.zero_()`
2. **Use torch.no_grad()** during inference to save memory
3. **Call .backward() only on scalars** (or provide gradient argument)
4. **Gradients accumulate** - this is a feature, not a bug!

### Next Steps

- **Next Tutorial**: `03_datasets.ipynb` - Learn about data loading
- **Practice**: Implement logistic regression using autograd
- **Challenge**: Implement a simple neural network from scratch

### Additional Resources

- [PyTorch Autograd Documentation](https://pytorch.org/docs/stable/autograd.html)
- [Autograd Mechanics](https://pytorch.org/docs/stable/notes/autograd.html)


## 🎯 Practice Exercises


In [ ]:
# Exercise 1: Compute gradient of f(x) = sin(x²) at x = 1
# Your code here:


# Exercise 2: Implement gradient descent for y = ax² + bx + c
# Find a, b, c given data points
# Your code here:


# Exercise 3: Compute the Jacobian matrix for a function f: R² → R²
# Your code here:


# Exercise 4: Implement custom loss function and compute its gradient
# Your code here:
